# Boxing Clever Solutions — Technical Operations Simulation
## Electronic Security Company (CCTV, Alarms, Access Control)

## Scenario
You've just joined as Technical Operations Intern. Boxing Clever installs and maintains electronic security systems across commercial sites in London and Essex. Client data is spread across three disconnected systems:

- **CRM** → `raw_clients.csv` — client accounts, contract types, monthly fees
- **Field Ops system** → `raw_installations.csv` — installation jobs, engineers, systems installed
- **Callout log** → `raw_callouts.csv` — reactive maintenance calls, fault types, SLA response

Nobody has ever consolidated these. Your job:
1. Load and audit each dataset
2. Clean and standardise
3. Consolidate into a master table using SQL
4. Engineer KPIs for the reporting framework
5. Export clean data ready for Power BI

---
> **Interview tip:** As you work through this, note every problem you find and every decision you make.
> That's your raw material for a STAR answer.

## Step 1 — Load and Audit the Raw Data

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

clients = pd.read_csv('raw_clients.csv')
installs = pd.read_csv('raw_installations.csv')
callouts = pd.read_csv('raw_callouts.csv')

print('=== CLIENTS ===')
print(clients.shape)
clients.head(8)

In [ ]:
# What problems exist in the CRM data?
print('Nulls in clients:')
print(clients.isnull().sum())
print()
print('client_id casing issues:')
print(clients['client_id'].str[:3].value_counts())
print()
print('Duplicate/variant client names (check for same site entered twice):')
name_lower = clients['client_name'].str.lower()
print(name_lower[name_lower.duplicated(keep=False)].sort_values().tolist())
print()
print('Mixed date formats:')
print(clients['contract_start'].head(10).tolist())
print()
print('Account manager casing issues:')
print(clients['account_manager'].value_counts())

In [ ]:
print('=== INSTALLATIONS ===')
print(installs.shape)
print()
print('Nulls:')
print(installs.isnull().sum())
print()
print('Job statuses:')
print(installs['status'].value_counts())
print()
print('Engineer name inconsistencies:')
print(installs['engineer'].value_counts())
installs.head(6)

In [ ]:
print('=== CALLOUTS ===')
print(callouts.shape)
print()
print('Nulls:')
print(callouts.isnull().sum())
print()
print('Priority breakdown:')
print(callouts['priority'].value_counts())
print()
print('Most common fault types:')
print(callouts['fault_type'].value_counts())
callouts.head(6)

## Step 2 — Clean and Standardise

Problems identified:
- `client_id` has mixed casing (CLT001 vs clt001)
- `client_name` has duplicate entries (same site, different capitalisation)
- `contract_start` has two date formats (DD/MM/YYYY and YYYY-MM-DD)
- `account_manager` has casing inconsistencies
- `monthly_fee_gbp` has nulls
- `engineer` names have casing inconsistencies across both installs and callouts
- `install_cost_gbp` has nulls
- `cameras_installed` is null for non-CCTV jobs

In [ ]:
clients_clean = clients.copy()

# Standardise IDs and names
clients_clean['client_id'] = clients_clean['client_id'].str.upper()
clients_clean['client_name'] = clients_clean['client_name'].str.strip().str.title()
clients_clean['account_manager'] = clients_clean['account_manager'].str.strip().str.title()

# Parse mixed date formats
def parse_date(d):
    for fmt in ('%d/%m/%Y', '%Y-%m-%d'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            continue
    return pd.NaT

clients_clean['contract_start'] = clients_clean['contract_start'].apply(parse_date)

# Fill missing monthly fees with median
median_fee = clients_clean['monthly_fee_gbp'].median()
clients_clean['monthly_fee_gbp'] = clients_clean['monthly_fee_gbp'].fillna(median_fee)

# Remove duplicates — same client name entered twice, keep highest fee
clients_clean['name_key'] = clients_clean['client_name'].str.lower().str.strip()
clients_clean = clients_clean.sort_values('monthly_fee_gbp', ascending=False).drop_duplicates(subset='name_key').drop(columns='name_key')

print(f'Clients: {len(clients)} → {len(clients_clean)} after dedup')
print('Remaining nulls:')
print(clients_clean.isnull().sum())
clients_clean.head(5)

In [ ]:
installs_clean = installs.copy()

# Standardise IDs and engineer names
installs_clean['client_ref'] = installs_clean['client_ref'].str.upper()
installs_clean['engineer'] = installs_clean['engineer'].str.strip().str.title()

# Parse dates
installs_clean['job_date'] = pd.to_datetime(installs_clean['job_date'])

# Fill missing install costs with median
installs_clean['install_cost_gbp'] = installs_clean['install_cost_gbp'].fillna(installs_clean['install_cost_gbp'].median())

# Fill cameras_installed with 0 where not applicable
installs_clean['cameras_installed'] = installs_clean['cameras_installed'].fillna(0).astype(int)

# Fill warranty
installs_clean['warranty_months'] = installs_clean['warranty_months'].fillna(12).astype(int)

print('Engineer names after standardisation:')
print(installs_clean['engineer'].value_counts())
print()
print('Job statuses:')
print(installs_clean['status'].value_counts())
installs_clean.head(5)

In [ ]:
callouts_clean = callouts.copy()

# Standardise
callouts_clean['client_id'] = callouts_clean['client_id'].str.upper()
callouts_clean['engineer'] = callouts_clean['engineer'].str.strip().str.title()

# Parse datetimes
for col in ['raised_at', 'responded_at', 'resolved_at']:
    callouts_clean[col] = pd.to_datetime(callouts_clean[col])

# Calculate response time and resolution time in hours
callouts_clean['response_hrs'] = (
    (callouts_clean['responded_at'] - callouts_clean['raised_at']).dt.total_seconds() / 3600
).round(2)

callouts_clean['resolution_hrs'] = (
    (callouts_clean['resolved_at'] - callouts_clean['raised_at']).dt.total_seconds() / 3600
).round(2)

# Flag SLA breaches
callouts_clean['sla_breached'] = callouts_clean['response_hrs'] > callouts_clean['sla_hrs']

print(f'SLA breaches: {callouts_clean["sla_breached"].sum()} / {len(callouts_clean)} ({callouts_clean["sla_breached"].mean()*100:.1f}%)')
print()
print('SLA breach rate by priority:')
print(callouts_clean.groupby('priority')['sla_breached'].mean().round(3).apply(lambda x: f'{x*100:.1f}%'))
callouts_clean.head(4)

## Step 3 — Consolidate with SQL

Load into SQLite and join across all three tables.

In [ ]:
conn = sqlite3.connect(':memory:')

clients_clean.to_sql('clients', conn, index=False, if_exists='replace')
installs_clean.to_sql('installations', conn, index=False, if_exists='replace')
callouts_clean.to_sql('callouts', conn, index=False, if_exists='replace')

# Verify join coverage
check = pd.read_sql('''
    SELECT COUNT(*) as matched
    FROM installations i
    JOIN clients c ON i.client_ref = c.client_id
''', conn)
print(f'Installation rows matching a client: {check["matched"].values[0]} / {len(installs_clean)}')

check2 = pd.read_sql('''
    SELECT COUNT(*) as matched
    FROM callouts co
    JOIN clients c ON co.client_id = c.client_id
''', conn)
print(f'Callout rows matching a client: {check2["matched"].values[0]} / {len(callouts_clean)}')

In [ ]:
# Master table: one row per client, with aggregated install and callout stats
master_query = '''
SELECT
    c.client_id,
    c.client_name,
    c.site_type,
    c.region,
    c.account_manager,
    c.contract_start,
    c.contract_type,
    c.monthly_fee_gbp,
    COALESCE(i.total_jobs, 0)              AS total_installs,
    COALESCE(i.completed_jobs, 0)          AS completed_installs,
    COALESCE(i.failed_jobs, 0)             AS failed_installs,
    COALESCE(i.total_install_revenue, 0)   AS total_install_revenue_gbp,
    COALESCE(i.total_cameras, 0)           AS total_cameras_installed,
    COALESCE(i.systems_installed, '')      AS systems_installed,
    COALESCE(co.total_callouts, 0)         AS total_callouts,
    COALESCE(co.sla_breaches, 0)           AS sla_breaches,
    COALESCE(co.repeat_faults, 0)          AS repeat_faults,
    co.avg_response_hrs                    AS avg_response_hrs,
    co.top_fault_type                      AS most_common_fault
FROM clients c
LEFT JOIN (
    SELECT
        client_ref,
        COUNT(*)                                              AS total_jobs,
        SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) AS completed_jobs,
        SUM(CASE WHEN status LIKE 'Failed%' THEN 1 ELSE 0 END) AS failed_jobs,
        ROUND(SUM(install_cost_gbp), 0)                       AS total_install_revenue,
        SUM(cameras_installed)                                AS total_cameras,
        GROUP_CONCAT(DISTINCT system_type)                    AS systems_installed
    FROM installations
    GROUP BY client_ref
) i ON c.client_id = i.client_ref
LEFT JOIN (
    SELECT
        client_id,
        COUNT(*)                                                  AS total_callouts,
        SUM(CASE WHEN sla_breached = 1 THEN 1 ELSE 0 END)        AS sla_breaches,
        SUM(CASE WHEN repeat_fault = 1 THEN 1 ELSE 0 END)        AS repeat_faults,
        ROUND(AVG(response_hrs), 1)                               AS avg_response_hrs,
        (SELECT fault_type FROM callouts c2
         WHERE c2.client_id = callouts.client_id
         GROUP BY fault_type ORDER BY COUNT(*) DESC LIMIT 1)      AS top_fault_type
    FROM callouts
    GROUP BY client_id
) co ON c.client_id = co.client_id
'''

master = pd.read_sql(master_query, conn)
print(f'Master table: {master.shape}')
master.head(6)

## Step 4 — Engineer KPI Metrics

These became Boxing Clever's **standard reporting framework**.

In [ ]:
kpi = master.copy()

# --- Revenue KPIs ---
kpi['annual_recurring_revenue_gbp'] = kpi['monthly_fee_gbp'] * 12

print('=== REVENUE KPIs ===')
print(f"Total Monthly Recurring Revenue: £{kpi['monthly_fee_gbp'].sum():,.0f}")
print(f"Total Annual Recurring Revenue:  £{kpi['annual_recurring_revenue_gbp'].sum():,.0f}")
print(f"Total Installation Revenue:      £{kpi['total_install_revenue_gbp'].sum():,.0f}")
print()

print('MRR by Contract Type:')
print(kpi.groupby('contract_type').agg(
    clients=('client_id','count'),
    total_mrr=('monthly_fee_gbp','sum'),
    avg_mrr=('monthly_fee_gbp','mean')
).round(0).sort_values('total_mrr', ascending=False))
print()

print('Revenue by Region:')
print(kpi.groupby('region')['monthly_fee_gbp'].sum().sort_values(ascending=False).apply(lambda x: f'£{x:,.0f}'))

In [ ]:
# --- Operational KPIs ---
print('=== OPERATIONAL KPIs ===')

total_callouts = kpi['total_callouts'].sum()
total_sla_breaches = kpi['sla_breaches'].sum()
sla_breach_rate = total_sla_breaches / total_callouts * 100 if total_callouts > 0 else 0

print(f'Total callouts: {int(total_callouts)}')
print(f'SLA breaches: {int(total_sla_breaches)} ({sla_breach_rate:.1f}%)')
print(f'Repeat faults: {int(kpi["repeat_faults"].sum())}')
print(f'Avg response time: {callouts_clean["response_hrs"].mean():.1f} hrs')
print()

# Installation completion rate
total_jobs = kpi['total_installs'].sum()
completed = kpi['completed_installs'].sum()
print(f'Installation completion rate: {completed/total_jobs*100:.1f}%')
print(f'Failed/revisit jobs: {int(kpi["failed_installs"].sum())}')
print()

# Engineer performance
print('Engineer callout workload:')
eng_kpi = callouts_clean.groupby('engineer').agg(
    callouts=('callout_id','count'),
    sla_breaches=('sla_breached','sum'),
    avg_response_hrs=('response_hrs','mean'),
    repeat_faults=('repeat_fault','sum')
).round(2)
eng_kpi['sla_breach_rate'] = (eng_kpi['sla_breaches'] / eng_kpi['callouts'] * 100).round(1).astype(str) + '%'
print(eng_kpi.sort_values('avg_response_hrs'))

In [ ]:
# --- At-Risk Client Flags ---
print('=== AT-RISK CLIENTS ===')

# Flag clients with high SLA breaches or repeat faults (churn risk)
kpi['sla_breach_rate'] = (kpi['sla_breaches'] / kpi['total_callouts'].replace(0, np.nan)).fillna(0)
kpi['high_repeat_faults'] = kpi['repeat_faults'] >= 2
kpi['high_sla_breach'] = kpi['sla_breach_rate'] > 0.4
kpi['at_risk'] = kpi['high_repeat_faults'] | kpi['high_sla_breach']

at_risk = kpi[kpi['at_risk']][['client_name','contract_type','monthly_fee_gbp',
                                'total_callouts','sla_breaches','repeat_faults','most_common_fault']]
print(f'{len(at_risk)} at-risk clients (repeated faults or high SLA breach rate):')
print(at_risk.sort_values('monthly_fee_gbp', ascending=False).to_string(index=False))

In [ ]:
# --- Account Manager KPIs ---
print('=== ACCOUNT MANAGER KPIs ===')
am_kpi = kpi.groupby('account_manager').agg(
    clients=('client_id','count'),
    total_mrr=('monthly_fee_gbp','sum'),
    total_install_revenue=('total_install_revenue_gbp','sum'),
    at_risk_clients=('at_risk','sum'),
    avg_callouts=('total_callouts','mean')
).round(1)
am_kpi['total_mrr'] = am_kpi['total_mrr'].apply(lambda x: f'£{x:,.0f}')
am_kpi['total_install_revenue'] = am_kpi['total_install_revenue'].apply(lambda x: f'£{x:,.0f}')
print(am_kpi)

## Step 5 — Export for Power BI

Three exports: the master table, a KPI summary, and the engineer performance table.

In [ ]:
# Master table (drives the main dashboards)
kpi.to_csv('boxing_clever_master.csv', index=False)

# KPI summary (for exec overview dashboard)
summary = pd.DataFrame({
    'KPI': ['Total MRR', 'Total ARR', 'Total Install Revenue', 'Total Clients',
             'SLA Breach Rate', 'At-Risk Clients', 'Install Completion Rate'],
    'Value': [
        f"£{kpi['monthly_fee_gbp'].sum():,.0f}",
        f"£{kpi['annual_recurring_revenue_gbp'].sum():,.0f}",
        f"£{kpi['total_install_revenue_gbp'].sum():,.0f}",
        str(len(kpi)),
        f'{sla_breach_rate:.1f}%',
        str(len(at_risk)),
        f'{completed/total_jobs*100:.1f}%'
    ]
})
summary.to_csv('boxing_clever_kpi_summary.csv', index=False)

print('Exports complete.')
print()
print(summary.to_string(index=False))

---
## Interview Debrief — STAR Answer

### Situation
"At Boxing Clever — an electronic security company doing CCTV, alarms, and access control — client data was split across three separate systems: the CRM, the field operations log, and the callout system. None of it had ever been joined up, so there was no single view of what revenue a client was generating, how many times engineers had been out, or whether SLAs were being met."

### Task
"My role was to consolidate those datasets and build a reporting framework that both the technical and commercial sides of the business could work from."

### Action
"I started by auditing each dataset — I found issues like inconsistent date formats, duplicate client entries, missing fee data, and engineer names entered in different cases across systems. I cleaned and standardised them in Python, then used SQL to join them into a single master table. From there I engineered the KPI metrics: recurring revenue by contract type, SLA breach rates by engineer and priority, repeat fault flags, and at-risk client indicators based on callout patterns. I built three Power BI dashboards from this — one for revenue, one for operational KPIs, and one for churn risk."

### Result
"The framework I built became the company's standard reporting structure. For the first time the commercial team could see which clients were generating the most revenue and which were generating the most operational cost — which directly informed how account managers prioritised renewals."

---
> **Numbers to remember:**
> - 3 source systems consolidated
> - Python + SQL for cleaning and joining
> - 3 Power BI dashboards (revenue, operational, churn/risk)
> - KPIs included: MRR/ARR, SLA breach rate, repeat fault rate, at-risk client flags, engineer performance
> - Output became the company's standard reporting framework